# packet_analyze_v2 — LLM 판정 (Colab T4 + vLLM)

로컬에서 `analyze.sh`로 만든 `report/*.evidence.json`을 LLM에 주고 판정(verdict)을 받는다.

**전제**: 런타임 → 유형 변경 → **T4 GPU** 선택.

흐름: repo clone → vLLM(Qwen2.5-14B-Instruct-AWQ) OpenAI 서버 부팅 → `llm_analyze.py` 실행

모델 선택 근거: T4(16GB VRAM) 기준 Qwen2.5-14B-Instruct-AWQ(~9GB)가 function-calling 정확도와
VRAM 여유 사이 최적점. 7B 대비 스키마 준수율·멀웨어 패밀리 식별력이 유의미하게 높음.

In [ ]:
# 1. repo clone
REPO = "https://github.com/DAADAISMYLIFE/packet_analyze_v2.git"
!git clone -q $REPO /content/packet_analyze_v2
%cd /content/packet_analyze_v2
!ls report/*.evidence.json

In [ ]:
# 2. 설치 (vLLM) — Colab T4는 CUDA 12.x이므로 CUDA 13 요구하는 최신 버전 배제
# vLLM 0.8.x = CUDA 12 호환 마지막 계열
!pip install -q "vllm==0.8.5"

In [ ]:
# 3. vLLM OpenAI 서버 백그라운드 부팅
#    Qwen2.5-14B-Instruct-AWQ: T4(16GB)에서 ~9GB 사용, function-calling 정확도 7B 대비 우수
#    --tool-call-parser hermes : Qwen2.5 function-calling 파서
import subprocess, os
os.makedirs('/content/logs', exist_ok=True)
srv = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-14B-Instruct-AWQ',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.90',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
], stdout=open('/content/logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('vLLM 부팅 중... (모델 다운로드 포함 5~8분)')

In [ ]:
# 4. 서버 준비 대기 (health 폴링)
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        print('✅ vLLM 준비 완료'); break
    except Exception:
        time.sleep(5)
        if i % 6 == 0: print(f'  대기 {i*5}s...')
else:
    print('❌ 타임아웃 — /content/logs/vllm.log 확인')
    !tail -30 /content/logs/vllm.log

In [ ]:
# 5. LLM 판정 — report/ 의 모든 evidence.json을 순서대로 처리
import glob, os

MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
evidence_files = sorted(glob.glob('report/*.evidence.json'))
print(f'evidence {len(evidence_files)}개 발견:')
for f in evidence_files:
    print(f'  {f}')
print()

for ev_path in evidence_files:
    name = os.path.basename(ev_path).replace('.evidence.json', '')
    print(f'\n{'='*60}')
    print(f'  분석: {name}')
    print(f'{'='*60}')
    !python scripts/llm_analyze.py {name} \
        --base-url http://localhost:8000/v1 \
        --model {MODEL} \
        --temperature 0.0

In [ ]:
# 6. verdict 전체 요약
import json, glob

for v_path in sorted(glob.glob('report/*.verdict.json')):
    name = os.path.basename(v_path).replace('.verdict.json', '')
    v = json.load(open(v_path))
    print(f'\n── {name}')
    print(f'   is_attack={v.get("is_attack")}  classification={v.get("classification")}  '
          f'confidence={v.get("confidence")}  needs_review={v.get("needs_review")}')
    print(f'   malware_family: {v.get("malware_family")}')
    ta = v.get('threat_actors', {})
    print(f'   victims: {ta.get("victim_ips",[])}  c2: {[c.get("ip") for c in ta.get("c2",[])]}') 
    print(f'   tool_trace: {len(v.get("tool_trace",[]))}회 / {v.get("rounds_used","?")}라운드')